## Feature Engineering

Feature Engineering is crucial in Machine Learning. It involves selecting, transforming, and creating features from raw data to improve model performance and interpretability. Effective feature engineering can significantly enhance the predictive power and generalization ability of machine learning models.

We know how to transfer numerical data, but...what happens if one or more of our feature columns are categorical? Machine leraning models can't deal with categorical columns. Therefore, we will have to cast the categorical values to numbers.

There basically two approaches that we can use depending on what kind of categorical columns we have:

* Nominal:
    - There is NO order between the labels
* Ordinal:
    - There IS an iimplicit order between the labels

For **nominal columns** we can use:

* Dummification:
    - We can use the [OneHotEncoder](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html) class available in sklearn for the corresponding column

For **ordinal columns** we can replace the labels by numbers respecting:

    - The implicit order
    - The relative order

to this end, we might want to use a Python dictionary in which the keys are the labels to replace, and the values, the numbers to use.

#### Loading and preparing the data

In [1]:
from sklearn.datasets import fetch_openml
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

from sklearn.preprocessing import MinMaxScaler, StandardScaler

In [2]:
X, y = fetch_openml("titanic", version=1, as_frame=True, return_X_y=True)

In [3]:
X

,pclass,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked,boat,body,home.dest
0,1,"Allen, Miss. Elisabeth Walton",female,29.0000,0,0,24160,211.3375,B5,S,2,NaN,"St Louis, MO"
1,1,"Allison, Master. Hudson Trevor",male,0.9167,1,2,113781,151.5500,C22 C26,S,11,NaN,"Montreal, PQ / Chesterville, ON"
2,1,"Allison, Miss. Helen Loraine",female,2.0000,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"
3,1,"Allison, Mr. Hudson Joshua Creighton",male,30.0000,1,2,113781,151.5500,C22 C26,S,NaN,135.0,"Montreal, PQ / Chesterville, ON"
4,1,"Allison, Mrs. Hudson J C (Bessie Waldo Daniels)",female,25.0000,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1304,3,"Zabour, Miss. Hileni",female,14.5000,1,0,2665,14.4542,NaN,C,NaN,328.0,NaN
1305,3,"Zabour, Miss. Thamine",female,NaN,1,0,2665,14.4542,NaN,C,NaN,NaN,NaN
1306,3,"Zakarian, Mr. Mapriededer",male,26.5000,0,0,2656,7.2250,NaN,C,NaN,304.0,NaN
1307,3,"Zakarian, Mr. Ortin",male,27.0000,0,0,2670,7.2250,NaN,C,NaN,NaN,NaN


# Pre-selecting features

Here, we can't use the passenger name because:

* Our model will depend on the passenger's name
* We will [overfit](https://www.ibm.com/topics/overfitting) our model

Therefore, we need to remove the passenger's name from our features "X"

In [4]:
X = X.drop(columns=['name'], axis=1)
X

,pclass,sex,age,sibsp,parch,ticket,fare,cabin,embarked,boat,body,home.dest
0,1,female,29.0000,0,0,24160,211.3375,B5,S,2,NaN,"St Louis, MO"
1,1,male,0.9167,1,2,113781,151.5500,C22 C26,S,11,NaN,"Montreal, PQ / Chesterville, ON"
2,1,female,2.0000,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"
3,1,male,30.0000,1,2,113781,151.5500,C22 C26,S,NaN,135.0,"Montreal, PQ / Chesterville, ON"
4,1,female,25.0000,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"
...,...,...,...,...,...,...,...,...,...,...,...,...
1304,3,female,14.5000,1,0,2665,14.4542,NaN,C,NaN,328.0,NaN
1305,3,female,NaN,1,0,2665,14.4542,NaN,C,NaN,NaN,NaN
1306,3,male,26.5000,0,0,2656,7.2250,NaN,C,NaN,304.0,NaN
1307,3,male,27.0000,0,0,2670,7.2250,NaN,C,NaN,NaN,NaN


# Categorical Nominal columns

Here, we can see some categorical nominal columns:

* sex
* embarked

In [5]:
X['embarked'].isna().sum()
rows_with_nas_in_embarked = X[ X['embarked'].isna() ].index
X = X.drop(rows_with_nas_in_embarked, axis= 0)
y = pd.DataFrame(y)
y = y.drop(rows_with_nas_in_embarked, axis= 0)
y = y['survived']

# Train Test split

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.20, random_state=0)

In [7]:
from sklearn.preprocessing import OneHotEncoder


ohe = OneHotEncoder(sparse_output=False)

ohe.fit(X_train[['sex','embarked']])
X_train_trans_np = ohe.transform(X_train[['sex','embarked']])
X_train_trans_np

array([[0., 1., 0., 0., 1.],
       [1., 0., 1., 0., 0.],
       [0., 1., 0., 1., 0.],
       ...,
       [0., 1., 0., 0., 1.],
       [0., 1., 0., 0., 1.],
       [1., 0., 0., 1., 0.]])

As we can see, the output is binary Numpy array that has more columns than our initial two columns. How we can have again a Pandas DataFrame with column names?

In [8]:
X_train_trans_df = pd.DataFrame(X_train_trans_np, columns=ohe.get_feature_names_out(), index=X_train.index)
X_train_trans_df

,sex_female,sex_male,embarked_C,embarked_Q,embarked_S
408,0.0,1.0,0.0,0.0,1.0
44,1.0,0.0,1.0,0.0,0.0
1072,0.0,1.0,0.0,1.0,0.0
786,0.0,1.0,0.0,0.0,1.0
1236,0.0,1.0,0.0,0.0,1.0
...,...,...,...,...,...
765,1.0,0.0,0.0,0.0,1.0
837,0.0,1.0,0.0,0.0,1.0
1218,0.0,1.0,0.0,0.0,1.0
561,0.0,1.0,0.0,0.0,1.0


Keep in mind that once our OneHotEncoder is fitted is fitted forever. We can use it to transform the test set **without** having to apply again the fit method.

In [9]:
X_test_trans_np = ohe.transform(X_test[['sex','embarked']])
X_test_trans_np

array([[0., 1., 0., 0., 1.],
       [0., 1., 0., 0., 1.],
       [0., 1., 0., 0., 1.],
       ...,
       [0., 1., 0., 0., 1.],
       [0., 1., 0., 0., 1.],
       [0., 1., 0., 0., 1.]])

In [10]:
X_test_trans_df = pd.DataFrame(X_test_trans_np, columns=ohe.get_feature_names_out(), index=X_test.index)
X_test_trans_df

,sex_female,sex_male,embarked_C,embarked_Q,embarked_S
611,0.0,1.0,0.0,0.0,1.0
414,0.0,1.0,0.0,0.0,1.0
530,0.0,1.0,0.0,0.0,1.0
1150,0.0,1.0,0.0,0.0,1.0
932,1.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...
388,1.0,0.0,0.0,0.0,1.0
612,1.0,0.0,0.0,0.0,1.0
753,0.0,1.0,0.0,0.0,1.0
555,0.0,1.0,0.0,0.0,1.0


# Combining data again

Once we transformer any categorical columns of our dataframe, we will have to combine again the numerical columns and hte transformed categorical columns back using `pd.concat()`.

However, we need to keep in mind one important thing. The values of the transformed columns will be either 0 or 1. Therefore, they will have a different range of value than the numerical ones!!! Then, we will have two options:

* transform again the numerical columns with a MinMaxScaler
* transform the whole dataset with another numerical scaler

# Saving our Encoder and transformers

As we're not going to train again the transformers or the encoders, we we can do is to save them on disk to use them later. How we can do this? We have two possible libraries to do it:

* pickle
* joblib

Let's see how to use the first one:

In [11]:
import pickle

with open("one_hot_encoder.pkl", "wb") as file:
    pickle.dump(ohe, file)

# Recovering a previously saved encoder

To recover the encoder from the disk, we can use another [context manager](https://realpython.com/python-with-statement/)

In [12]:
with open("one_hot_encoder.pkl", "rb") as file:
    ohe2 = pickle.load(file)

Once we have loaded the encoder, we can use it to transform any new data that we might want to predict.

```python
X_new_trans_np = ohe2.transform(X_new)
X_new_trans_df = pd.DataFrame(X_new_trans_np, columns=ohe.get_feature_names_out(), index=X_new.index)
X_new_trans_df
````

**WARNING!!!!**

The encoder remembers **which columns and in which order** they were used to fit it. Therefore, whenever we want to use it again, X_new MUST have the **same columns, and in the same order** that dataset used to train the encoder.